# Capstone: end-to-end credit portfolio workflow

**Prerequisites:** complete the overview path through `07_advanced_quant/margin_collateral_and_xva.ipynb`. You should be fluent with core types, market data, instrument pricing, portfolio construction, scenarios, P&L attribution, financial statement modeling, performance analytics, and risk analytics.

This notebook ties every layer into one linear story: **market snapshot → price credit instruments → compute risk metrics → aggregate a portfolio → stress the market → decompose P&L → model an issuer → measure performance → explain model outputs for reporting**.


## Concept: an integrated credit portfolio workflow

Real credit desks rarely use a single API in isolation. A typical internal workflow:

1. **Market setup** — Build a `MarketContext` with discount curves, forward curves, **hazard curves**, and FX for a single as-of date.
2. **Credit instrument pricing** — Price a corporate bond (with `credit_curve_id`), a CDS, and a term loan via `price_instrument_with_metrics`.
3. **Risk metrics** — Extract DV01, CS01, modified duration, and spread measures per instrument.
4. **Portfolio construction** — Aggregate positions across two entities with `value_portfolio`.
5. **Scenario revaluation** — Apply a +50 bp credit stress with `apply_scenario_and_revalue`, then drill down to per-instrument repricing.
6. **P&L attribution** — Decompose a one-day MTM change into carry, rates, and credit factors using waterfall attribution.
7. **Issuer modeling** — Build a simple P&L bridge (revenue → EBITDA → pre-tax) for credit surveillance.
8. **Performance & risk analytics** — *(disconnected demo)* — Synthetic price-path analysis for CAGR, Sharpe, drawdowns, beta, and ruin probability.
9. **Reporting** — Trace dependencies and explain formulas for a modeled line item.

The code below is **self-contained** (no external data files). Run cells **top to bottom**.


## Step 1 — Market setup

Construct a **`MarketContext`** with:
- **USD OIS** discount curve
- **USD SOFR** forward curve (for floating-rate instruments)
- **ACME-HAZARD** — investment-grade hazard curve (entity: ACME Corp)
- **GLOBEX-HAZARD** — high-yield hazard curve (entity: GLOBEX Inc)
- **EUR/USD** spot via `FxMatrix`

Serialize with `to_json()` so all downstream calls share one market snapshot.


In [1]:
import json
from datetime import date

import sys
sys.path.insert(0, "..")

from _shared import DEMO_AS_OF
from finstack_quant.core.currency import Currency
from finstack_quant.core.market_data import (
    DiscountCurve,
    ForwardCurve,
    FxMatrix,
    HazardCurve,
    MarketContext,
)

# Shared example as-of date (2025-01-15) used across the notebook suite.
AS_OF = DEMO_AS_OF
AS_OF_STR = AS_OF.isoformat()

mc = MarketContext()

ois = DiscountCurve(
    "USD-OIS",
    AS_OF,
    [(0.0, 1.0), (0.25, 0.9875), (0.5, 0.975), (1.0, 0.95),
     (2.0, 0.90), (5.0, 0.75), (10.0, 0.50)],
    day_count="act_365f",
)
mc.insert(ois)

sofr = ForwardCurve(
    "USD-SOFR",
    0.25,
    knots=[(0.0, 0.045), (1.0, 0.048), (2.0, 0.05), (5.0, 0.052)],
    base_date=AS_OF,
    day_count="act_360",
)
mc.insert(sofr)

acme_hazard = HazardCurve(
    "ACME-HAZARD",
    AS_OF,
    [(0.5, 0.010), (1.0, 0.012), (2.0, 0.015),
     (3.0, 0.018), (5.0, 0.022), (10.0, 0.028)],
    recovery_rate=0.40,
    par_spreads=[(0.5, 60.166784), (1.0, 68.399472), (2.0, 79.233874),
                 (3.0, 89.070795), (5.0, 104.841082), (10.0, 129.818769)],
)
mc.insert(acme_hazard)

globex_hazard = HazardCurve(
    "GLOBEX-HAZARD",
    AS_OF,
    [(0.5, 0.035), (1.0, 0.040), (2.0, 0.048),
     (3.0, 0.055), (5.0, 0.062), (10.0, 0.070)],
    recovery_rate=0.35,
    par_spreads=[(0.5, 228.264839), (1.0, 250.550944), (2.0, 281.090527),
                 (3.0, 306.178070), (5.0, 339.361439), (10.0, 379.679140)],
)
mc.insert(globex_hazard)

fx = FxMatrix()
fx.set_quote(Currency("EUR"), Currency("USD"), 1.08)
mc.insert_fx(fx)

market_json = mc.to_json()
print(f"MarketContext JSON: {len(market_json):,} chars")
print(f"As-of: {AS_OF_STR}")
print(f"ACME  5Y survival: {acme_hazard.sp(5.0):.6f}  (IG)")
print(f"GLOBEX 5Y survival: {globex_hazard.sp(5.0):.6f}  (HY)")


MarketContext JSON: 3,949 chars
As-of: 2025-01-15
ACME  5Y survival: 0.915761  (IG)
GLOBEX 5Y survival: 0.767590  (HY)


## Step 2 — Credit instrument pricing

Three credit instruments across two entities:

| Instrument | Entity | Type | Model |
|---|---|---|---|
| **ACME-CORP-5Y** | ACME | Fixed-coupon corporate bond with `credit_curve_id` | `discounting` |
| **CDS-GLOBEX-5Y** | GLOBEX | 5Y CDS (protection buyer) | `hazard_rate` |
| **ACME-TL-B** | ACME | Term loan (fixed spread, amortizing) | `discounting` |


In [2]:
from finstack_quant.valuations import ValuationResult
from finstack_quant.valuations.instruments import price_instrument_with_metrics, validate_instrument_json

acme_bond = json.dumps({
    "type": "bond",
    "spec": {
        "id": "ACME-CORP-5Y",
        "notional": {"amount": "1000000", "currency": "USD"},
        "issue_date": "2024-01-15",
        "maturity": "2029-01-15",
        "discount_curve_id": "USD-OIS",
        "credit_curve_id": "ACME-HAZARD",
        "cashflow_spec": {
            "Fixed": {
                "coupon_type": "Cash",
                "rate": 0.05,
                "freq": {"count": 6, "unit": "months"},
                "dc": "Thirty360",
                "bdc": "following",
                "calendar_id": "weekends_only",
                "stub": "None",
                "end_of_month": False,
                "payment_lag_days": 0,
            }
        },
        "call_put": None,
        "attributes": {"tags": ["credit"], "meta": {"sector": "Industrials"}},
        "pricing_overrides": {},
    },
})

globex_cds = json.dumps({
    "type": "credit_default_swap",
    "spec": {
        "id": "CDS-GLOBEX-5Y",
        "notional": {"amount": 5000000.0, "currency": "USD"},
        "side": "pay",
        "convention": "isda_na",
        "premium": {
            "start": "2025-03-20",
            "end": "2030-03-20",
            "frequency": {"count": 3, "unit": "months"},
            "stub": "ShortFront",
            "bdc": "following",
            "calendar_id": None,
            "day_count": "Act360",
            "spread_bp": 250.0,
            "discount_curve_id": "USD-OIS",
        },
        "protection": {
            "credit_curve_id": "GLOBEX-HAZARD",
            "recovery_rate": 0.35,
            "settlement_delay": 3,
        },
        "pricing_overrides": {},
        "attributes": {"tags": ["credit"], "meta": {"sector": "Technology"}},
    },
})

acme_loan = json.dumps({
    "type": "term_loan",
    "spec": {
        "id": "ACME-TL-B",
        "notional_limit": {"amount": "10000000", "currency": "USD"},
        "currency": "USD",
        "issue_date": "2024-01-15",
        "maturity": "2029-01-15",
        "discount_curve_id": "USD-OIS",
        "rate": {"Fixed": {"rate_bp": 550}},
        "day_count": "Act360",
        "frequency": {"count": 3, "unit": "months"},
        "bdc": "modified_following",
        "calendar_id": None,
        "stub": "None",
        "amortization": {"PercentPerPeriod": {"bp": 250}},
        "coupon_type": "Cash",
        "settlement_days": 2,
        "attributes": {"tags": ["credit", "loan"], "meta": {"sector": "Industrials"}},
        "pricing_overrides": {},
    },
})

instruments = [
    ("ACME bond", acme_bond, "discounting"),
    ("GLOBEX CDS", globex_cds, "hazard_rate"),
    ("ACME term loan", acme_loan, "discounting"),
]
for label, inst_json, model in instruments:
    validate_instrument_json(inst_json)
    out = price_instrument_with_metrics(inst_json, market_json, AS_OF_STR, model=model)
    vr = ValuationResult.from_json(out)
    print(f"{label:<16} PV = {vr.price:>14,.2f} {vr.currency}")


ACME bond        PV =     944,073.90 USD
GLOBEX CDS       PV =     174,540.73 USD
ACME term loan   PV =   9,049,783.87 USD


## Step 3 — Risk metrics

Request DV01, CS01, modified duration, and spread measures per instrument. The metrics registry returns `None` for metrics that don't apply to a given instrument type.


In [3]:
metric_requests = {
    "ACME bond": (acme_bond, "discounting", ["dv01", "duration_mod", "convexity", "z_spread", "cs01"]),
    "GLOBEX CDS": (globex_cds, "hazard_rate", ["cs01", "cs01_hazard", "jump_to_default"]),
    "ACME term loan": (acme_loan, "discounting", ["dv01", "effective_rate"]),
}

for label, (inst_json, model, mets) in metric_requests.items():
    out = price_instrument_with_metrics(
        inst_json, market_json, AS_OF_STR, model=model, metrics=mets,
    )
    vr = ValuationResult.from_json(out)
    print(f"— {label}")
    for m in mets:
        v = vr.get_metric(m)
        if v is not None:
            print(f"    {m}: {v}")
    print()


— ACME bond
    dv01: -341.94181993504753
    duration_mod: 3.5461947827626767
    convexity: 0.14989445112140604
    z_spread: 0.009917851826660069
    cs01: -339.90900193847483



— GLOBEX CDS
    cs01: 1865.772334863752
    cs01_hazard: 1190.823900717951
    jump_to_default: 3250000.0

— ACME term loan
    dv01: -2736.5517701096833



## Step 4 — Portfolio construction

Aggregate the three credit positions across two entities (ACME, GLOBEX) using `value_portfolio`. The valuation JSON reports `total_base_ccy` and a per-entity breakdown.


In [4]:
from finstack_quant.portfolio import PortfolioValuation, value_portfolio

portfolio = json.dumps({
    "id": "credit-portfolio-2025",
    "as_of": AS_OF_STR,
    "base_ccy": "USD",
    "entities": {
        "ACME": {"id": "ACME"},
        "GLOBEX": {"id": "GLOBEX"},
    },
    "positions": [
        {
            "position_id": "POS-ACME-BOND",
            "entity_id": "ACME",
            "instrument_id": "ACME-CORP-5Y",
            "instrument_spec": json.loads(acme_bond),
            "quantity": 1.0,
            "unit": "units",
        },
        {
            "position_id": "POS-GLOBEX-CDS",
            "entity_id": "GLOBEX",
            "instrument_id": "CDS-GLOBEX-5Y",
            "instrument_spec": json.loads(globex_cds),
            "quantity": 1.0,
            "unit": "units",
        },
        {
            "position_id": "POS-ACME-TL",
            "entity_id": "ACME",
            "instrument_id": "ACME-TL-B",
            "instrument_spec": json.loads(acme_loan),
            "quantity": 1.0,
            "unit": "units",
        },
    ],
})

val_json = value_portfolio(portfolio, market_json)

# Typed wrapper: base-currency total without digging through the JSON envelope.
base_valuation = PortfolioValuation.from_json(val_json)
print(f"Portfolio total PV: {base_valuation.total_value:,.2f} {base_valuation.base_ccy}\n")

# The per-entity and per-position breakdowns live in the raw envelope.
val_obj = json.loads(val_json)
print("By entity:")
for eid, money in val_obj.get("by_entity", {}).items():
    print(f"  {eid}: {float(money['amount']):>14,.2f} {money['currency']}")
print(f"\nPositions valued: {len(val_obj['position_values'])}")


Portfolio total PV: 10,168,398.50 USD

By entity:
  ACME:   9,993,857.78 USD
  GLOBEX:     174,540.73 USD

Positions valued: 3


## Step 5 — Scenario revaluation

Apply a **+50 bp par-CDS shock** to both hazard curves and revalue the portfolio. The `par_cds` curve kind bumps quoted CDS spreads, which the engine maps into hazard space using the discount curve as anchor.

`portfolio.apply_scenario_and_revalue` is the canonical path: it shocks the market and re-values **every position** in one call, returning the stressed `PortfolioValuation` plus the scenario `ApplicationReport`. Comparing its total against the base total from Step 4 gives the portfolio stress P&L.


In [5]:
from finstack_quant.portfolio import apply_scenario_and_revalue
from finstack_quant.scenarios import build_scenario_spec, list_builtin_templates

print(f"Built-in templates: {list_builtin_templates()}\n")

stress_ops = json.dumps([
    {
        "kind": "curve_parallel_bp",
        "curve_kind": "par_cds",
        "curve_id": "ACME-HAZARD",
        "discount_curve_id": "USD-OIS",
        "bp": 50.0,
    },
    {
        "kind": "curve_parallel_bp",
        "curve_kind": "par_cds",
        "curve_id": "GLOBEX-HAZARD",
        "discount_curve_id": "USD-OIS",
        "bp": 50.0,
    },
])
scenario_json = build_scenario_spec(
    "credit-stress-50bp",
    stress_ops,
    name="Credit widening +50bp par-CDS",
)

# One call: shock the market and revalue every position in the portfolio.
stressed_val_json, report_json = apply_scenario_and_revalue(
    portfolio, scenario_json, market_json,
)
stressed_valuation = PortfolioValuation.from_json(stressed_val_json)
report = json.loads(report_json)

print(f"Authored operations: {report['user_operations']}")
print(f"Applied operations:  {report['operations_applied']}")
if report.get("warnings"):
    print("Warnings:", report["warnings"])

stress_pnl = stressed_valuation.total_value - base_valuation.total_value
ccy = stressed_valuation.base_ccy
print(f"\n{'Base portfolio PV':<22} {base_valuation.total_value:>16,.2f} {ccy}")
print(f"{'Stressed portfolio PV':<22} {stressed_valuation.total_value:>16,.2f} {ccy}")
print(f"{'Stress P&L':<22} {stress_pnl:>+16,.2f} {ccy}")


Built-in templates: ['gfc_2008', 'covid_2020', 'rate_shock_2022', 'svb_2023', 'ltcm_1998']



Authored operations: 2
Applied operations:  2

Base portfolio PV         10,168,398.50 USD
Stressed portfolio PV     10,250,143.03 USD
Stress P&L                   +81,744.53 USD


### Step 5b — Drill-down: where the portfolio number comes from

The portfolio number above is the number a risk report quotes. To *explain* it, drop one level: build the same stressed market with `scenarios.apply_scenario_to_market` and reprice each instrument on its own. Because every position holds one unit, the per-instrument deltas sum back to the portfolio stress P&L — a reconciliation, not an alternative calculation.


In [6]:
from finstack_quant.scenarios import apply_scenario_to_market
from finstack_quant.valuations.instruments import price_instrument

stressed_market = apply_scenario_to_market(scenario_json, market_json, AS_OF_STR)["market_json"]

print(f"{'Instrument':<18} {'Base PV':>14} {'Stressed PV':>14} {'Δ P&L':>12}")
print("-" * 62)
drilldown_pnl = 0.0
for label, inst_json, model in instruments:
    base_out = price_instrument(inst_json, market_json, AS_OF_STR, model=model)
    stress_out = price_instrument(inst_json, stressed_market, AS_OF_STR, model=model)
    pv_base = float(json.loads(base_out)["value"]["amount"])
    pv_stress = float(json.loads(stress_out)["value"]["amount"])
    drilldown_pnl += pv_stress - pv_base
    print(f"{label:<18} {pv_base:>14,.2f} {pv_stress:>14,.2f} {pv_stress - pv_base:>+12,.2f}")
print("-" * 62)
print(f"{'Sum of positions':<18} {'':>14} {'':>14} {drilldown_pnl:>+12,.2f}")
print(f"{'Portfolio stress P&L':<18} {'':>14} {'':>14} {stress_pnl:>+12,.2f}")
print(f"\nReconciles: {abs(drilldown_pnl - stress_pnl) < 0.01}")


Instrument                Base PV    Stressed PV        Δ P&L
--------------------------------------------------------------
ACME bond              944,073.90     926,941.81   -17,132.10
GLOBEX CDS             174,540.73     273,417.35   +98,876.63
ACME term loan       9,049,783.87   9,049,783.87        +0.00
--------------------------------------------------------------
Sum of positions                                   +81,744.53
Portfolio stress P&L                                 +81,744.53

Reconciles: True


## Step 6 — P&L attribution

Decompose a one-day MTM change on the ACME bond into **carry**, **rates**, and **credit** factors using **waterfall** attribution. Between T₀ and T₁:

- Discount rates shift **+5 bp**
- Credit spreads (hazard) widen **+10 bp**
- One day of carry accrues


In [7]:
from finstack_quant.attribution import PnlAttribution, attribute_pnl

AS_OF_T0 = "2025-01-15"
AS_OF_T1 = "2025-01-16"
base_t0 = date.fromisoformat(AS_OF_T0)
base_t1 = date.fromisoformat(AS_OF_T1)

mc_t0 = MarketContext()
mc_t0.insert(DiscountCurve(
    "USD-OIS", base_t0,
    [(0.0, 1.0), (0.25, 0.9875), (0.5, 0.975), (1.0, 0.95),
     (2.0, 0.90), (5.0, 0.75), (10.0, 0.50)],
    day_count="act_365f",
))
mc_t0.insert(HazardCurve(
    "ACME-HAZARD", base_t0,
    [(0.5, 0.010), (1.0, 0.012), (2.0, 0.015),
     (3.0, 0.018), (5.0, 0.022), (10.0, 0.028)],
    recovery_rate=0.40,
))
market_t0 = mc_t0.to_json()

mc_t1 = MarketContext()
mc_t1.insert(DiscountCurve(
    "USD-OIS", base_t1,
    [(0.0, 1.0), (0.25, 0.9872), (0.5, 0.9744), (1.0, 0.9489),
     (2.0, 0.8983), (5.0, 0.7468), (10.0, 0.4963)],
    day_count="act_365f",
))
mc_t1.insert(HazardCurve(
    "ACME-HAZARD", base_t1,
    [(0.5, 0.011), (1.0, 0.013), (2.0, 0.016),
     (3.0, 0.019), (5.0, 0.023), (10.0, 0.029)],
    recovery_rate=0.40,
))
market_t1 = mc_t1.to_json()

waterfall_method = {"Waterfall": ["Carry", "RatesCurves", "CreditCurves"]}
attr_json = attribute_pnl(acme_bond, market_t0, market_t1, AS_OF_T0, AS_OF_T1, waterfall_method)
attr = PnlAttribution.from_json(attr_json)

print(attr.explain())
print(f"\nResidual within tolerance: {attr.residual_within_tolerance()}")
print(f"Repricings: {attr.num_repricings}")


Total P&L: USD 20177.40
  ├─ Carry: USD 25144.24 (124.6%)
  │   ├─ Coupon Income: USD 25138.89
  │   ├─ Pull-to-Par: USD 39.58
  │   ├─ Roll-Down: USD -34.23
  ├─ Rates Curves: USD -2980.31 (-14.8%)
  ├─ Credit Curves: USD -1986.52 (-9.8%)
  └─ Residual: USD 0.00 (0.0%)

Residual within tolerance: True
Repricings: 7


## Step 7 — Issuer modeling

Build a minimal **ACME** P&L spec: explicit drivers (`revenue`, `cogs`, `opex`, `interest`) and computed nodes (`gross_profit`, `ebitda`, `ebt`). The shared example fixture `_shared.demo_pl_model` wraps the `ModelBuilder` calls used across the notebook suite — every formula still lives in the model, not in Python. `Evaluator` produces a `StatementResult`; `to_pandas_wide()` scans all line items across quarters.


In [8]:
import sys
sys.path.insert(0, "..")

from _shared import demo_pl_model
from finstack_quant.statements import Evaluator

# Shared quarterly P&L fixture: revenue/cogs/opex drivers plus in-model
# gross_profit and ebitda formulas. ACME-specific drivers are passed in;
# interest and the pre-tax bridge are the credit-surveillance overlay.
spec = demo_pl_model(
    "acme-pl",
    periods=["2025Q1", "2025Q2", "2025Q3", "2025Q4"],
    revenue=[250.0, 260.0, 270.0, 280.0],
    cogs=[150.0, 155.0, 160.0, 165.0],
    opex=[50.0, 52.0, 54.0, 56.0],
    actual_through="2025Q2",
    margins=False,
    values={"interest": [10.0, 10.0, 10.0, 10.0]},
    computes={"ebt": "ebitda - interest"},
)
result = Evaluator().evaluate(spec)
wide = result.to_pandas_wide()
print(wide)


period_id     2025Q1  2025Q2  2025Q3  2025Q4
interest        10.0    10.0    10.0    10.0
opex            50.0    52.0    54.0    56.0
cogs           150.0   155.0   160.0   165.0
revenue        250.0   260.0   270.0   280.0
gross_profit   100.0   105.0   110.0   115.0
ebitda          50.0    53.0    56.0    59.0
ebt             40.0    43.0    46.0    49.0


## Step 8 — Performance & risk analytics *(disconnected demo)*

> **Note:** This section uses a **synthetic random price path** — it is *not* connected to the portfolio instruments above. It demonstrates the `Performance` and risk analytics APIs in isolation.

Simulate a 252-day equity path, compute CAGR / Sharpe / drawdown, then estimate factor beta and ruin probability.


In [9]:
import random
from datetime import timedelta

from finstack_quant.analytics import Performance

random.seed(42)
dates = [date(2024, 1, 1) + timedelta(days=i) for i in range(252)]
prices = [100.0]
bench_prices = [100.0]
for _ in range(251):
    prices.append(prices[-1] * (1 + random.gauss(0.0003, 0.012)))
    bench_prices.append(bench_prices[-1] * (1 + random.gauss(0.0004, 0.01)))

perf = Performance.from_arrays(
    dates,
    [prices, bench_prices],
    ["PORTFOLIO", "BENCH"],
    benchmark_ticker="BENCH",
)
print(f"CAGR:    {perf.cagr()[0]:.6f}")
print(f"Sharpe:  {perf.sharpe()[0]:.6f}")
print(f"Sortino: {perf.sortino()[0]:.6f}")
print(f"Ann vol: {perf.volatility()[0]:.6f}")
print(f"VaR 95%: {perf.value_at_risk(0.95)[0]:.6f}")
print(f"ES  95%: {perf.expected_shortfall(0.95)[0]:.6f}")
print(f"Max DD:  {perf.max_drawdown()[0]:.6f}")
print(f"\nBeta vs benchmark: {perf.beta()[0].beta:.4f}")


CAGR:    0.312332
Sharpe:  1.076175
Sortino: 1.598518
Ann vol: 0.191232
VaR 95%: -0.020447
ES  95%: -0.024872
Max DD:  -0.155293

Beta vs benchmark: -0.1121


## Step 9 — Reporting

Export the model spec and evaluation result to JSON, then use `DependencyTracer` for a dependency tree and `explain_formula` for a period-specific walkthrough of how **EBITDA** is computed.


In [10]:
from finstack_quant.statements_analytics import DependencyTracer, explain_formula

model_json = spec.to_json()
eval_json = result.to_json()

deps = DependencyTracer(model_json).dependency_tree("ebitda")
print("EBITDA dependencies:")
print(deps)

explanation = explain_formula(model_json, eval_json, "ebitda", "2025Q1")
print("\nFormula explanation (2025Q1):")
print(explanation)

EBITDA dependencies:
ebitda (gross_profit - opex)
gross_profit (revenue - cogs)
revenue
cogs
opex


Formula explanation (2025Q1):
{'node_id': 'ebitda', 'period_id': '2025Q1', 'final_value': 50.0, 'node_type': 'Calculated', 'formula_text': 'gross_profit - opex', 'breakdown': [{'component': 'gross_profit', 'value': 100.0, 'operation': None}, {'component': 'opex', 'value': 50.0, 'operation': None}]}


## Takeaways

You walked the full credit portfolio stack:

1. **Market data** — Discount, forward, and **hazard** curves shared by all pricers.
2. **Credit instruments** — Bond (with `credit_curve_id`), CDS, and term loan across two entities.
3. **Risk metrics** — DV01, CS01, duration, z-spread per instrument.
4. **Portfolio** — Entity-level aggregation via `value_portfolio`.
5. **Scenarios** — Par-CDS stress (+50 bp) revalued portfolio-wide via `apply_scenario_and_revalue`, reconciled against a per-instrument drill-down.
6. **P&L attribution** — Waterfall decomposition into carry, rates, and credit.
7. **Issuer modeling** — P&L bridge for credit surveillance.
8. **Performance & risk** — CAGR, Sharpe, beta, ruin probability (synthetic demo).
9. **Reporting** — Dependency tracing and formula explanation.

From here, extend with calibrated curves, more entities, real position data, and wire scenario results into limits and reporting — using the same JSON-first contracts throughout.